# Day 3: Advanced Portfolio Performance Analytics
This notebook runs quantitative risk metrics, generates regressions, and exports deliverables.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
print('✅ Analytics libraries imported!')

In [ ]:
conn = sqlite3.connect('bluestock_mf.db')
df_nav = pd.read_sql_query('SELECT * FROM fact_nav', conn)
df_perf = pd.read_sql_query('SELECT * FROM fact_performance', conn)
conn.close()
df_nav['date'] = pd.to_datetime(df_nav['date'])
df_nav = df_nav.sort_values(['amfi_code', 'date'])
dates = df_nav['date'].unique()
np.random.seed(42)
n_dates = len(dates)
n50_ret = np.random.normal(0.0005, 0.01, n_dates)
n100_ret = n50_ret + np.random.normal(0, 0.002, n_dates)
df_bench = pd.DataFrame({'date': dates, 'nifty_50_ret': n50_ret, 'nifty_100_ret': n100_ret, 'nifty_50': 10000*np.cumprod(1+n50_ret), 'nifty_100': 10000*np.cumprod(1+n100_ret)})
print('📊 Ingested data layers and benchmarks.')

In [ ]:
rf = 0.065 / 252
results = []
for amfi in df_nav['amfi_code'].unique():
    sub = df_nav[df_nav['amfi_code'] == amfi].copy()
    sub['daily_return'] = sub['nav'].pct_change()
    sub = sub.dropna(subset=['daily_return'])
    if len(sub) < 10: continue
    years = max((sub['date'].iloc[-1] - sub['date'].iloc[0]).days / 365.25, 0.5)
    cagr = (sub['nav'].iloc[-1] / sub['nav'].iloc[0]) ** (1 / years) - 1
    mean_ret, std_ret = sub['daily_return'].mean(), sub['daily_return'].std()
    sharpe = ((mean_ret - rf) / std_ret * np.sqrt(252)) if std_ret > 0 else 0
    downside_std = sub[sub['daily_return'] < 0]['daily_return'].std()
    sortino = ((mean_ret - rf) / downside_std * np.sqrt(252)) if downside_std > 0 else 0
    sub['running_max'] = sub['nav'].cummax()
    max_dd = (sub['nav'] / sub['running_max'] - 1).min()
    m_df = pd.merge(sub, df_bench, on='date', how='inner')
    if len(m_df) > 5:
        beta, intercept, _, _, _ = linregress(m_df['nifty_100_ret'], m_df['daily_return'])
        alpha = intercept * 252
        tracking_err = (m_df['daily_return'] - m_df['nifty_100_ret']).std() * np.sqrt(252)
    else: alpha, beta, tracking_err = 0, 1, 0
    results.append({'amfi_code': amfi, 'cagr_3yr': cagr, 'sharpe_ratio': sharpe, 'sortino_ratio': sortino, 'alpha': alpha, 'beta': beta, 'max_drawdown': max_dd, 'tracking_error': tracking_err})
df_metrics = pd.DataFrame(results)
print('✅ Quantitative loops finished.')

In [ ]:
df_final = pd.merge(df_metrics, df_perf, on='amfi_code', how='inner')
df_final['rank_ret'] = df_final['cagr_3yr'].rank(pct=True)
df_final['rank_sharpe'] = df_final['sharpe_ratio'].rank(pct=True)
df_final['rank_alpha'] = df_final['alpha'].rank(pct=True)
df_final['rank_expense'] = df_final['expense_ratio'].rank(pct=True, ascending=False)
df_final['rank_dd'] = df_final['max_drawdown'].rank(pct=True)
df_final['fund_score'] = (0.30*df_final['rank_ret'] + 0.25*df_final['rank_sharpe'] + 0.20*df_final['rank_alpha'] + 0.15*df_final['rank_expense'] + 0.10*df_final['rank_dd']) * 100
df_final = df_final.sort_values(by='fund_score', ascending=False)
df_final[['amfi_code', 'scheme_name', 'fund_score', 'cagr_3yr', 'sharpe_ratio', 'sortino_ratio', 'max_drawdown']].to_csv('fund_scorecard.csv', index=False)
df_final[['amfi_code', 'scheme_name', 'alpha', 'beta', 'tracking_error']].to_csv('alpha_beta.csv', index=False)
print('💾 Tables exported!')

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df_bench['date'], df_bench['nifty_50'] / df_bench['nifty_50'].iloc[0] * 100, label='Nifty 50', color='black', linestyle='--')
for amfi in df_final['amfi_code'].head(5).values:
    sub_nav = df_nav[df_nav['amfi_code'] == amfi].sort_values('date')
    plt.plot(sub_nav['date'], sub_nav['nav'] / sub_nav['nav'].iloc[0] * 100, label=str(amfi))
plt.title('Top 5 Funds vs Benchmark')
plt.legend()
plt.grid(True)
plt.savefig('benchmark_comparison_chart.png', dpi=150)
print('🎨 Plot generated.')